In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# 1. Device Agnostic Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

In [ ]:
# 2. Data Preprocessing & Augmentation
# Note: Dataset contains a mix of grayscale and color images.
# transforms.Lambda ensures all inputs are strictly converted to 3-channel RGB.
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.Lambda(lambda img: img.convert('RGB')),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Lambda(lambda img: img.convert('RGB')),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [ ]:
# 3. Load Dataset using ImageFolder
data_dir = "./CovidCT_Scan"
image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ['train', 'test']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=16, shuffle=(x == 'train'), num_workers=2)
    for x in ['train', 'test']
}

class_names = image_datasets['train'].classes  # Expected: ['covid', 'normal']
print(f"Loaded Classes: {class_names}")

In [ ]:
# 4. Initialize Pretrained Model (ResNet18)
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Modify final fully connected layer for binary classification (2 classes)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Low learning rate for fine-tuning

In [ ]:
# 5. Training and Validation Loop
num_epochs = 2  # 1-2 epochs is sufficient for fine-tuning
best_acc = 0.0

start_time = time.time()
for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")
    for phase in ['train', 'test']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])

        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

        if phase == 'test' and epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), 'covid_model.pth')

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed // 60:.0f}m {elapsed % 60:.0f}s")
print(f"Best Test Accuracy: {best_acc:.2%}")